In [1]:
# Imports
from google.colab import drive
import os
import sys
import joblib
from tqdm import tqdm #add a visual progress bar to loops
import scipy.sparse as sp
import pandas as pd
import numpy as np
import nltk
from nltk.corpus import stopwords
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
import torch
from transformers import AutoModel, AutoTokenizer

In [8]:
# Install dependencies from requirements.txt
print("Installing dependencies...")
!pip install -r requirements.txt

Installing dependencies...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 21.6 MB/s eta 0:00:00


In [2]:
# Downloads
nltk.download("punkt")
nltk.download('punkt_tab')

nltk.download('stopwords')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [3]:
# Remove limit for diplaying columns
pd.set_option('display.max_colwidth', None)

In [4]:
# Mount drive, if needed
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
# Path to Google Drive project directory
drive_project_dir = '/content/drive/MyDrive/twitter-multimodal-hate-speech-classifier'

# Set project directory as current working directory
os.chdir(drive_project_dir)

# Set path to directory for data files in Google Drive
drive_data_dir = 'data' # relative path
os.makedirs(drive_data_dir, exist_ok=True)

# Set path to directory where trained models are stored in Google Drive
drive_models_dir = 'models'
os.makedirs(drive_models_dir, exist_ok=True)

In [9]:
# Import functions from .py file
from src.preprocessing_utils import extract_timestamp_features, combine_text_columns, text_preprocessor, text_tokenizer, extract_roberta_embeddings


In [8]:
# Load train, validation and test data from previously stored parquet
data_train = pd.read_parquet(os.path.join(drive_data_dir, 'train.parquet'))
data_val = pd.read_parquet(os.path.join(drive_data_dir, 'val.parquet'))
data_test = pd.read_parquet(os.path.join(drive_data_dir, 'test.parquet'))

In [9]:
# Check the format of data
data_train.head(3)

,file_id,img_url,labels,tweet_url,tweet_text,labels_str,created_at,img_text,img_caption,target,num_symbols_origmsg
1,1063020048816660480,http://pbs.twimg.com/ext_tw_video_thumb/1063019643709747200/pu/img/wK5HgoX6tFfxWJmi.jpg,"[5, 5, 5]",https://twitter.com/user/status/1063020048816660480,My horses are retarded https://t.co/HYhqc6d5WN,"[OtherHate, OtherHate, OtherHate]",2018-11-15 10:45:04.252,,a screenshot of a man in a snowy forest,5,46
2,1108927368075374593,http://pbs.twimg.com/media/D2OzhzHUwAADQjd.jpg,"[0, 0, 0]",https://twitter.com/user/status/1108927368075374593,“NIGGA ON MA MOMMA YOUNGBOY BE SPITTING REAL SHIT NIGGA” https://t.co/UczofqHrLq,"[NotHate, NotHate, NotHate]",2019-03-22 03:04:22.080,,a man getting his hair cut by a barber,0,80
3,1114558534635618305,http://pbs.twimg.com/ext_tw_video_thumb/1114018095348084738/pu/img/m6oP5fBm2RCcq5Wn.jpg,"[1, 0, 0]",https://twitter.com/user/status/1114558534635618305,RT xxSuGVNGxx: I ran into this HOLY NIGGA TODAY 😭😭😭😭 https://t.co/Wa6Spl9kIw,"[Racist, NotHate, NotHate]",2019-04-06 16:00:36.810,,a young man riding a skateboard down a sidewalk,0,76


In [10]:
# Define lists of feature groups
text_cols = ['tweet_text', 'img_text', 'img_caption']
time_col = ['created_at']
msg_length_col = ['num_symbols_origmsg']

feature_cols = text_cols + time_col + msg_length_col

# Time feature names (4 columns returned by timestamp extractor)
time_feature_names = ['hour_sin', 'hour_cos', 'is_weekend', 'month']

# Feature name of aggregated message (three text fields - tweet, text on image, image caption)
msg_length_feature_names = ['num_symbols_origmsg']

In [11]:
# Define X_train, X_val, X_test
X_train = data_train[feature_cols]
X_val = data_val[feature_cols]
X_test = data_test[feature_cols]

In [12]:
# Check the format of data
X_train.head(3)

,tweet_text,img_text,img_caption,created_at,num_symbols_origmsg
1,My horses are retarded https://t.co/HYhqc6d5WN,,a screenshot of a man in a snowy forest,2018-11-15 10:45:04.252,46
2,“NIGGA ON MA MOMMA YOUNGBOY BE SPITTING REAL SHIT NIGGA” https://t.co/UczofqHrLq,,a man getting his hair cut by a barber,2019-03-22 03:04:22.080,80
3,RT xxSuGVNGxx: I ran into this HOLY NIGGA TODAY 😭😭😭😭 https://t.co/Wa6Spl9kIw,,a young man riding a skateboard down a sidewalk,2019-04-06 16:00:36.810,76


## Preprocessing setup
(setup for RoBERTa is in corresponding section)

In [13]:
# Define stop-words to remove from text
en_stopwords = stopwords.words('english')

In [14]:
# Pipeline to handle timestamps
timestamp_pipeline = Pipeline([
    ('time_extractor', FunctionTransformer(extract_timestamp_features)),
    ('scaler', StandardScaler())
])

In [15]:
# Pipeline to handle texts:
  # 1. merge three text columns,
  # 2. preprocess text
  # 3. vectorize applying Bag of Words (BoW)
text_pipeline_bow = Pipeline(
    [
        ('text_merger', FunctionTransformer(combine_text_columns)),  # concatenates all thre text fields of the tweet (message test, text from posted image, verbal descr. of image) into one text
        ('vectorizer', CountVectorizer(
            preprocessor=text_preprocessor,
            tokenizer=text_tokenizer,
            stop_words=en_stopwords,
            lowercase=False,
            min_df=5, # Ignore words that appear in less than 5 messages
            max_df=0.85, # Ignore words that appear in more than 85% of all messages
            max_features=10000),
         ),
    ]
)

In [16]:
# Pipeline to handle texts:
  # 1. merge three text columns,
  # 2. preprocess text
  # 3. vectorize applying TF-IDF
text_pipeline_tfidf = Pipeline(
    [
        ('text_merger', FunctionTransformer(combine_text_columns)),  # concatenates all thre text fields of the tweet (message test, text from posted image, verbal descr. of image) into one text
        ('vectorizer', TfidfVectorizer(
            preprocessor=text_preprocessor,
            tokenizer=text_tokenizer,
            stop_words=en_stopwords,
            lowercase=False,
            min_df=5, # Ignore words that appear in less than 5 messages
            max_df=0.85, # Ignore words that appear in more than 85% of all messages
            max_features=10000),
         ),
    ]
)

In [17]:
# Pipeline to handle texts:
  # 1. merge three text columns,
  # 2. preprocess text
  # 3. vectorize applying TF-IDF with ngrams
text_pipeline_tfidf_ngram = Pipeline(
    [
        ('text_merger', FunctionTransformer(combine_text_columns)),  # concatenates all thre text fields of the tweet (message test, text from posted image, verbal descr. of image) into one text
        ('vectorizer', TfidfVectorizer(
            preprocessor=text_preprocessor,
            tokenizer=text_tokenizer,
            stop_words=en_stopwords,
            lowercase=False,
            ngram_range=(1, 2),
            min_df=5, # Ignore words that appear in less than 5 messages
            max_df=0.85, # Ignore words that appear in more than 85% of all messages
            max_features=10000),
         ),
    ]
)

In [19]:
# Pipeline to handle length (in symbols) of an original tweet text message
msg_length_pipeline = Pipeline([
    ('scaler', StandardScaler())
])

In [20]:
# Define preprocessor containing BoW
preprocessor_bow = ColumnTransformer(
    transformers=[
        ('text_features', text_pipeline_bow, text_cols),
        ('time_features', timestamp_pipeline, time_col),
        ('msg_length_features', msg_length_pipeline, msg_length_col)
    ],
    remainder='drop'
)

In [21]:
# Define preprocessor containing TF-IDF
preprocessor_tfidf = ColumnTransformer(
    transformers=[
        ('text_features', text_pipeline_tfidf, text_cols),
        ('time_features', timestamp_pipeline, time_col),
        ('msg_length_features', msg_length_pipeline, msg_length_col)
    ],
    remainder='drop'
)

In [22]:
# Define preprocessor containing TF-IDF with ngrams
preprocessor_tfidf_ngram = ColumnTransformer(
    transformers=[
        ('text_features', text_pipeline_tfidf_ngram, text_cols),
        ('time_features', timestamp_pipeline, time_col),
        ('msg_length_features', msg_length_pipeline, msg_length_col)
    ],
    remainder='drop'
)

## Preprocess + Bag of Words (BoW):

In [ ]:
# Fit preprocessor object that used BoW on train data
preprocessor_bow = preprocessor_bow.fit(X_train)

/usr/local/lib/python3.12/dist-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/feature_extraction/text.py:402: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ["'", 'abov', 'ani', 'becaus', 'befor', 'doe', 'dure', 'onc', 'onli', 'ourselv', 'themselv', 'veri', 'whi', 'yourselv'] not in stop_words.
  warnings.warn(


In [ ]:
# Save fitted preprocessor object
joblib.dump(preprocessor_bow, os.path.join(drive_models_dir, 'fitted_preprocessor_bow.joblib'))

['/content/drive/MyDrive/ML_course/Final-Project/models/fitted_preprocessor_bow.joblib']

In [ ]:
# Number of words in the vocabulary
len(preprocessor_bow.named_transformers_['text_features'].named_steps['vectorizer'].vocabulary_)

10000

In [ ]:
# Transform train, validation and test data applying fitted preprocessor
X_train_bow_processed_matrix =  preprocessor_bow.transform(X_train)
X_val_bow_processed_matrix =  preprocessor_bow.transform(X_val)
X_test_bow_processed_matrix =  preprocessor_bow.transform(X_test)

In [ ]:
# Save processed data as sparse matrix for later use
sp.save_npz(os.path.join(drive_data_dir, 'X_train_bow_processed_matrix.npz'), X_train_bow_processed_matrix)
sp.save_npz(os.path.join(drive_data_dir, 'X_val_bow_processed_matrix.npz'), X_val_bow_processed_matrix)
sp.save_npz(os.path.join(drive_data_dir, 'X_test_bow_processed_matrix.npz'), X_test_bow_processed_matrix)

In [ ]:
# Derive all feature names for preprocessed data and save them separately
  # Text feature names (words/stems learned by CountVectorizer)
text_feature_names_bow = list(preprocessor_bow.named_transformers_['text_features'].named_steps['vectorizer'].get_feature_names_out())

  # Combine all feature names
all_feature_names_bow = text_feature_names_bow + time_feature_names + msg_length_feature_names

# Save feature names
joblib.dump(all_feature_names_bow, os.path.join(drive_data_dir, 'feature_names_bow_processed.joblib'))

['/content/drive/MyDrive/ML_course/Final-Project/data/feature_names_bow_processed.joblib']

## Preprocess + TF-IDF:

In [ ]:
# Fit preprocessor object that applies TF-IDF
preprocessor_tfidf = preprocessor_tfidf.fit(X_train)

/usr/local/lib/python3.12/dist-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/feature_extraction/text.py:402: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ["'", 'abov', 'ani', 'becaus', 'befor', 'doe', 'dure', 'onc', 'onli', 'ourselv', 'themselv', 'veri', 'whi', 'yourselv'] not in stop_words.
  warnings.warn(


In [ ]:
# Save fitted preprocessor object - uncomment to perform saving
joblib.dump(preprocessor_tfidf, os.path.join(drive_models_dir, 'fitted_preprocessor_tfidf.joblib'))

['/content/drive/MyDrive/ML_course/Final-Project/models/fitted_preprocessor_tfidf.joblib']

In [ ]:
# Check number of words in the vocabulary
len(preprocessor_tfidf.named_transformers_['text_features'].named_steps['vectorizer'].vocabulary_)

10000

In [ ]:
# Transform train, validation and test data applying fitted preprocessor (TF-IDF)
X_train_tfidf_processed_matrix =  preprocessor_tfidf.transform(X_train)
X_val_tfidf_processed_matrix =  preprocessor_tfidf.transform(X_val)
X_test_tfidf_processed_matrix =  preprocessor_tfidf.transform(X_test)

In [ ]:
# Save processed data as sparse matrix for later use
sp.save_npz(os.path.join(drive_data_dir, 'X_train_tfidf_processed_matrix.npz'), X_train_tfidf_processed_matrix)
sp.save_npz(os.path.join(drive_data_dir, 'X_val_tfidf_processed_matrix.npz'), X_val_tfidf_processed_matrix)
sp.save_npz(os.path.join(drive_data_dir, 'X_test_tfidf_processed_matrix.npz'), X_test_tfidf_processed_matrix)

In [ ]:
# Derive all feature names for preprocessed data and save them separately
  # Text feature names (words/stems learned by TF-IDF)
text_feature_names_tfidf = list(preprocessor_tfidf.named_transformers_['text_features'].named_steps['vectorizer'].get_feature_names_out())

  # Combine all feature names
all_feature_names_tfidf = text_feature_names_tfidf + time_feature_names + msg_length_feature_names

# Save feature names
joblib.dump(all_feature_names_tfidf, os.path.join(drive_data_dir, 'feature_names_tfidf_processed.joblib'))

['/content/drive/MyDrive/ML_course/Final-Project/data/feature_names_tfidf_processed.joblib']

## Process + TF-IDF (ngram)

In [ ]:
# Fit preprocessor object that applies TF-IDF
preprocessor_tfidf_ngram = preprocessor_tfidf_ngram.fit(X_train)

/usr/local/lib/python3.12/dist-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/feature_extraction/text.py:402: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ["'", 'abov', 'ani', 'becaus', 'befor', 'doe', 'dure', 'onc', 'onli', 'ourselv', 'themselv', 'veri', 'whi', 'yourselv'] not in stop_words.
  warnings.warn(


In [ ]:
# Save fitted preprocessor object - uncomment to perform saving
joblib.dump(preprocessor_tfidf_ngram, os.path.join(drive_models_dir, 'fitted_preprocessor_tfidf_ngram.joblib'))

['/content/drive/MyDrive/ML_course/Final-Project/models/fitted_preprocessor_tfidf_ngram.joblib']

In [ ]:
# Check number of words in the vocabulary
len(preprocessor_tfidf_ngram.named_transformers_['text_features'].named_steps['vectorizer'].vocabulary_)

10000

In [ ]:
# Transform train, validation and test data applying fitted preprocessor (TF-IDF)
X_train_tfidf_ngram_processed_matrix =  preprocessor_tfidf_ngram.transform(X_train)
X_val_tfidf_ngram_processed_matrix =  preprocessor_tfidf_ngram.transform(X_val)
X_test_tfidf_ngram_processed_matrix =  preprocessor_tfidf_ngram.transform(X_test)

In [ ]:
# Save processed data as sparse matrix for later use
sp.save_npz(os.path.join(drive_data_dir, 'X_train_tfidf_ngram_processed_matrix.npz'), X_train_tfidf_ngram_processed_matrix)
sp.save_npz(os.path.join(drive_data_dir, 'X_val_tfidf_ngram_processed_matrix.npz'), X_val_tfidf_ngram_processed_matrix)
sp.save_npz(os.path.join(drive_data_dir, 'X_test_tfidf_ngram_processed_matrix.npz'), X_test_tfidf_ngram_processed_matrix)

In [ ]:
# Derive all feature names for preprocessed data and save them separately
  # Text feature names (words/stems learned by CountVectorizer)
text_feature_names_tfidf_ngram = list(preprocessor_tfidf_ngram.named_transformers_['text_features'].named_steps['vectorizer'].get_feature_names_out())

  # Combine all feature names
all_feature_names_tfidf_ngram = text_feature_names_tfidf_ngram + time_feature_names + msg_length_feature_names

# Save feature names
joblib.dump(all_feature_names_tfidf_ngram, os.path.join(drive_data_dir, 'feature_names_tfidf_ngram_processed.joblib'))

['/content/drive/MyDrive/ML_course/Final-Project/data/feature_names_tfidf_ngram_processed.joblib']

## Preprocess + RoBERTa

In [14]:
# Initialize Hugging Face tokenizer and model
MODEL_NAME = "cardiffnlp/twitter-roberta-base"
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME).to(device)
model.eval()

config.json:   0%|          | 0.00/565 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  501MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  501MB            

[transformers] RobertaModel LOAD REPORT from: cardiffnlp/twitter-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.decoder.bias      | UNEXPECTED |  | 
lm_head.decoder.weight    | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors: downloading bytes:           |  0.00B            

RobertaModel(
  (embeddings): RobertaEmbeddings(
    (word_embeddings): Embedding(50265, 768, padding_idx=1)
    (token_type_embeddings): Embedding(1, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
    (position_embeddings): Embedding(514, 768, padding_idx=1)
  )
  (encoder): RobertaEncoder(
    (layer): ModuleList(
      (0-11): 12 x RobertaLayer(
        (attention): RobertaAttention(
          (self): RobertaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): RobertaSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
            (dropou

In [15]:
text_pipeline_roberta = Pipeline(
    [
        ('text_merger', FunctionTransformer(combine_text_columns)),
        ('vectorizer', FunctionTransformer(
            extract_roberta_embeddings,
            kw_args={
                'model': model,
                'tokenizer': tokenizer,
                'device': device
            }
        )),
    ]
)

In [17]:
# Define preprocessor containing Twitter-RoBERTa
preprocessor_roberta = ColumnTransformer(
    transformers=[
        ('text_features', text_pipeline_roberta, text_cols),
        ('time_features', timestamp_pipeline, time_col),
        ('msg_length_features', msg_length_pipeline, msg_length_col)
    ],
    remainder='drop'
)

In [ ]:
# Fit Preprocessor on train data
%%time
preprocessor_roberta = preprocessor_roberta.fit(X_train)

In [ ]:
# Save fitted preprocessor object - uncomment to repeat saving
joblib.dump(preprocessor_roberta, os.path.join(drive_models_dir, 'fitted_preprocessor_roberta.joblib'))

In [ ]:
# Transform train data applying fitted preprocessor (RoBERTa)
%%time
X_train_roberta_processed_matrix =  preprocessor_roberta.transform(X_train)

In [ ]:
# Transform validation applying fitted preprocessor (RoBERTa)
%%time
X_val_roberta_processed_matrix =  preprocessor_roberta.transform(X_val)

100%|██████████| 132/132 [00:24<00:00,  5.32it/s]


In [ ]:
# Transform test data applying fitted preprocessor (RoBERTa)
X_test_roberta_processed_matrix =  preprocessor_roberta.transform(X_test)

100%|██████████| 263/263 [00:56<00:00,  4.66it/s]


In [ ]:
X_test_roberta_processed_matrix.shape

(8411, 773)

In [ ]:
X_test_roberta_processed_matrix

array([[ 0.0762899 ,  0.06986839,  0.19109699, ..., -0.640628  ,
        -0.45360256, -0.69556008],
       [ 0.10962658, -0.09297065,  0.04410898, ...,  1.5609683 ,
        -0.45360256, -0.34010922],
       [ 0.05910939, -0.03697312,  0.08913834, ...,  1.5609683 ,
        -0.78077514, -0.19792888],
       ...,
       [ 0.06965351, -0.07091097,  0.07496373, ..., -0.640628  ,
        -0.78077514, -0.12683871],
       [ 0.03287124, -0.04864006,  0.08180368, ..., -0.640628  ,
        -0.78077514, -1.19319127],
       [ 0.08116765,  0.02744846,  0.06855287, ..., -0.640628  ,
         0.85508775,  1.72150575]])

In [ ]:
# Save processed data matrix (dense) for later use
sp.save_npz(os.path.join(drive_data_dir, 'X_train_roberta_processed_matrix.npz'), X_train_roberta_processed_matrix)
sp.save_npz(os.path.join(drive_data_dir, 'X_val_roberta_processed_matrix.npz'), X_val_roberta_processed_matrix)
sp.save_npz(os.path.join(drive_data_dir, 'X_test_roberta_processed_matrix.npz'), X_test_roberta_processed_matrix)